In [1]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path="requant/vabc/f16_requant_zs_vabc_VGGNet.tflite")
interpreter.allocate_tensors()

for tensor in interpreter.get_tensor_details():
    print(f"Name: {tensor['name']}, Type: {tensor['dtype']}")

Name: serving_default_Input_Layer:0, Type: <class 'numpy.float32'>
Name: vgg16_bn/GlobalAvgPool/Mean/reduction_indices, Type: <class 'numpy.int32'>
Name: vgg16_bn/Conv1-1/BiasAdd/ReadVariableOp, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv2-1/BiasAdd/ReadVariableOp, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv3-1/BiasAdd/ReadVariableOp, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv3-2/BiasAdd/ReadVariableOp, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv4-1/BiasAdd/ReadVariableOp, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv4-2/BiasAdd/ReadVariableOp, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv1-1/Conv2D, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv2-1/Conv2D, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv3-1/Conv2D, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv3-2/Conv2D, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv4-1/Conv2D, Type: <class 'numpy.float16'>
Name: vgg16_bn/Conv4-2/Conv2D, Type: <class 'numpy.float16'>
Name: vgg16_bn/FC-2/Bias

In [8]:
import tensorflow.lite as tflite

# Load the TFLite model
model_path = "requant/vabc/f16_requant_zs_vabc_VGGNet.tflite"
interpreter = tflite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Get the total memory needed
arena_size = interpreter.get_tensor_details()[-1]['index']
print(f"Estimated Tensor Arena Size: {arena_size} bytes")

Estimated Tensor Arena Size: 102 bytes


In [2]:
import tensorflow.lite as tflite

interpreter = tflite.Interpreter(model_path=model_path)
ops = interpreter._get_ops_details()

In [3]:
for x in ops:
    print(x)

{'index': 0, 'op_name': 'PAD', 'inputs': array([0, 3]), 'outputs': array([38])}
{'index': 1, 'op_name': 'CONV_2D', 'inputs': array([38, 28, 12]), 'outputs': array([39])}
{'index': 2, 'op_name': 'PAD', 'inputs': array([39,  2]), 'outputs': array([40])}
{'index': 3, 'op_name': 'MAX_POOL_2D', 'inputs': array([40]), 'outputs': array([41])}
{'index': 4, 'op_name': 'CONV_2D', 'inputs': array([41, 11, 13]), 'outputs': array([42])}
{'index': 5, 'op_name': 'CONV_2D', 'inputs': array([41, 29, 14]), 'outputs': array([43])}
{'index': 6, 'op_name': 'CONV_2D', 'inputs': array([43, 10, 15]), 'outputs': array([44])}
{'index': 7, 'op_name': 'ADD', 'inputs': array([42, 44]), 'outputs': array([45])}
{'index': 8, 'op_name': 'CONV_2D', 'inputs': array([45,  9, 16]), 'outputs': array([46])}
{'index': 9, 'op_name': 'CONV_2D', 'inputs': array([46,  8, 17]), 'outputs': array([47])}
{'index': 10, 'op_name': 'ADD', 'inputs': array([45, 47]), 'outputs': array([48])}
{'index': 11, 'op_name': 'CONV_2D', 'inputs': a

In [ ]:
def convert_tflite_to_c(tflite_path, c_path):
    with open(tflite_path, 'rb') as tflite_file:
        tflite_content = tflite_file.read()

    lines = [', '.join([f'0x{byte:02x}' for byte in tflite_content[i:i+12]]) for i in range(0, len(tflite_content), 12)]
    array = ',\n  '.join(lines)

    with open(c_path, 'w') as header_file:
        header_file.write('const unsigned char model[] = {\n  ')
        header_file.write(f'{array}\n')
        header_file.write('};\n\n')

In [ ]:
convert_tflite_to_c('requant/vabc/requant_zs_augmented_vabc_VGGNet.tflite', 'c_req_aug_vabc_VGGNet.h')

In [3]:
import sys

def convert_to_c_array(file_path):
    with open(file_path, 'rb') as f:
        data = f.read()
    hex_array = ', '.join(f'0x{b:02x}' for b in data)
    c_array = f'const unsigned char model_tflite[] = {{ {hex_array} }};\n'
    c_array += f'const unsigned int model_tflite_len = {len(data)};'
    return c_array

c_code = convert_to_c_array('requant/vabc/requant_zs_vabc_VGGNet.tflite')
with open("c_req_vabc_VGGNet.h", 'w') as f:
    f.write(c_code)